In [1]:
%pip install -q pandas langchain langchain-openai langchain-tavily langchain-core

import os, csv, datetime
from typing import List, Dict, Optional
import pandas as pd
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch  # 검색 툴
from langchain_core.tools import tool
from langchain.agents import create_agent

load_dotenv()

Note: you may need to restart the kernel to use updated packages.


True

In [2]:
# -------------------------
# 1) 파일(CSV) 툴 2개
# -------------------------
@tool("csv_write", return_direct=False)
def csv_write(path: str, rows: List[Dict], mode: str = "a") -> str:
    """
    rows(list[dict])를 CSV 파일에 저장/추가한다.
    - path: csv 경로 (예: 'news.csv')
    - rows: 딕셔너리 목록. 키가 헤더가 됨. 새 파일이면 헤더를 자동 생성
    - mode: 'a' 추가(기본), 'w' 새로쓰기
    반환: 작성 행 수/헤더 정보
    """
    if not rows:
        return "rows가 비어있습니다."
    # 헤더 순서를 일정하게(가독성 좋은 기본 필드 우선)
    priority = ["date", "query", "title", "url", "source", "snippet"]
    keys = list({k for r in rows for k in r.keys()})
    ordered = priority + [k for k in keys if k not in priority]

    file_exists = os.path.isfile(path)
    write_header = (not file_exists) or (mode == "w")
    with open(path, mode, newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=ordered, extrasaction="ignore")
        if write_header:
            w.writeheader()
        for r in rows:
            w.writerow(r)
    return f"CSV 저장 완료: {path}, 행 수={len(rows)}, 헤더={ordered}"

@tool("csv_head", return_direct=False)
def csv_head(path: str, n: int = 5) -> str:
    """
    CSV 상위 n행 미리보기. (헤더 포함)
    """
    try:
        df = pd.read_csv(path)
        return df.head(n).to_string(index=False)
    except FileNotFoundError:
        return f"파일 없음: {path}"
    except Exception as e:
        return f"CSV 읽기 오류: {e}"



In [3]:
# -------------------------
# 2) 검색 툴 (Tavily)
# -------------------------
tavily = TavilySearch(max_results=5)

In [4]:
# -------------------------
# 3) 프롬프트 & Agent
# -------------------------
SYSTEM_PROMPT = """너는 '웹검색'과 'CSV 저장' 두 가지 툴만 사용하는 어시스턴트다.
- 최신 뉴스나 웹결과는 Tavily 검색 툴로 가져온다.
- 사용자가 파일로 저장하라고 하면, 결과를 적절한 필드(date, query, title, url, source, snippet)로 정규화하여 csv_write를 호출한다.
- 파일 저장 후에는 csv_head로 상위 5행을 보여줘서 확인하게 한다.
- 한국어로 간결히 답하되, 마지막에 한 줄 결론을 덧붙여라.
"""




In [9]:
llm = ChatOpenAI(model='gpt-5.4-mini', temperature=0)
TOOLS = [tavily, csv_write, csv_head]

agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=SYSTEM_PROMPT
)

In [11]:
print("\n=== 예시 A: 뉴스 요약만 ===")
result = agent.invoke({'messages':[{'role':'user', 'content':'오늘 SK하이닉스 뉴스 5개만 요약해줘.'}]})
result




=== 예시 A: 뉴스 요약만 ===


{'messages': [HumanMessage(content='오늘 SK하이닉스 뉴스 5개만 요약해줘.', additional_kwargs={}, response_metadata={}, id='4933c0c6-6519-4c5a-8bbd-d1260400acbe'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 1576, 'total_tokens': 1627, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DtjcZAZBIQFBHmEOFFEhxhgsLpAHi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ef1f4-49bb-7873-9e35-6fffb4e87b27-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '오늘 SK하이닉스 뉴스 5개', 'search_depth': 'advanced', 'topic': 'news', 'time_range': 'day', 'include_images': False}, 'id': 'call_HVgzw0dS2CjxMSMImeH3svVv'

In [7]:
# (B) 검색 → CSV로 저장(한 번에)
print("\n=== 예시 B: 검색해서 news.csv로 저장 ===")
today = datetime.date.today().isoformat()
q = f"{today} AI 주요 뉴스 5개를 검색해서 news.csv로 저장해줘."
result = agent.invoke({"messages": [{"role": "user", "content": q}]})
print(result["messages"][-1].content)



=== 예시 B: 검색해서 news.csv로 저장 ===
AI 주요 뉴스 5개가 news.csv로 저장되었습니다. 아래는 저장된 내용의 상위 5행입니다:

|    date    |    query    |                                                                                          title                                                                                          |                                                   url                                                    |      source      |                                                                                                           snippet                                                                                                           |
|------------|-------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------------------|------------------|--------

In [8]:
# (C) 이미 저장된 파일 미리보기
print("\n=== 예시 C: 저장 확인 ===")
result = agent.invoke({"messages": [{"role": "user", "content": "저장된 news.csv 상위 5행을 보여줘."}]})
print(result["messages"][-1].content)



=== 예시 C: 저장 확인 ===
저장된 `news.csv`의 상위 5행은 다음과 같습니다:

|      date    |    query    |                                                                 title                                                                 |                                                                 url                                                                 |     source     |                                                                                                                                                       snippet                                                                                                                                                       |
|--------------|-------------|---------------------------------------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------------------------------------------------------------------|---------------

[실습]
- 오늘 IT 대기업 인수 뉴스 5개 찾아서 it_news.csv로 저장하고 미리보기
- 엔비디아(또는 삼성전자) 관련 최신 기사 5개만 요약하고 nvidia_{오늘날짜}.csv로 저장
- 방금 만든 파일 상위 3행만 확인하기

In [ ]:
print('\n=== 검색해서 it_news.csv로 저장 ===')
q2 = f'{today} 오늘 IT 대기업 인수 뉴스 5개 찾아서 it_news.csv로 저장해줘.'
result2 = agent.invoke({'messages':[{'role':'user', 'content':q2}]})
print(result2['messages'][-1].content)


=== 검색해서 news.csv로 저장 ===
저장 완료했습니다. `it_news.csv`에 5건을 넣었고, 상위 5행도 확인했습니다.  
다만 현재 검색 결과에는 **“IT 대기업 인수(M&A)”에 정확히 해당하는 기사만 5개가 충분히 잡히지 않아**, 일부는 AI/빅테크 관련 거래·투자성 기사로 포함되었습니다.

원하시면 제가 다시 **“인수·합병만”** 기준으로 좁혀서 5개를 재검색해 드릴게요.  
결론: `it_news.csv` 저장은 완료됐고, 원하시면 M&A 기사만 다시 추려드릴 수 있습니다.


In [15]:
print('\n=== 검색해서 nvidia_260623.csv로 저장 ===')
q3 = f'{today} 엔비디아 관련 최신 기사 5개만 요약해서 nvidia_260623.csv로 저장해줘.'
result3 = agent.invoke({'messages':[{'role':'user', 'content':q3}]})
print(result3['messages'][-1].content)


=== 검색해서 nvidia_260623.csv로 저장 ===
요청하신 5개 기사 요약을 `nvidia_260623.csv`로 저장했습니다.  
다만 `csv_head`는 저장 직후 파일을 찾지 못해 미리보기를 보여주지 못했습니다.

저장된 내용은 엔비디아 관련 최신 기사 5건을 `date, query, title, url, source, snippet` 형식으로 정규화해 넣었습니다. 필요하시면 제가 다시 확인용 미리보기를 시도해드릴게요.

결론: `nvidia_260623.csv` 저장은 완료됐고, 미리보기는 도구상 바로 확인되지 않았습니다.


In [18]:
print('\n=== it_news.csv 상위 3행 ===')
print(csv_head.invoke({'path': 'it_news.csv', 'n': 3}))

print('\n=== nvidia_260623.csv 상위 3행 ===')
print(csv_head.invoke({'path': 'nvidia_260623.csv', 'n': 3}))


=== it_news.csv 상위 3행 ===
      date                                                                      query                                                          title                                                                                                                     url  source                                                                                                                              snippet
2026-06-22 2026-06-23 today IT big tech acquisition news 5 deals acquisitions mergers    Google DeepMind signs AI research deal with film studio A24  https://www.reuters.com/business/media-telecom/google-deepmind-signs-ai-research-deal-with-film-studio-a24-2026-06-22/ Reuters Google DeepMind and A24 announced an AI research partnership; A24 gets access to DeepMind research, infrastructure and global reach.
2026-06-22 2026-06-23 today IT big tech acquisition news 5 deals acquisitions mergers   AI startup Reflection signs computing power deal with SpaceX http